# 03 — Correlation and Lag Analysis

**Project:** SERSA Export Activity and Industrial Consumption Dynamics  
**Author:** Cesar Enrique Banda Martinez  
**Date:** 2026  

---

## Context

Notebook 02 established three key preliminary findings:

1. **Different structural dynamics:** Trade flows are stationary with seasonal cycles; SERSA revenue shows sustained growth. Raw level correlations will be misleading.
2. **Opposite seasonal patterns:** Trade exports peak in August while SERSA revenue peaks in January — seasonality is not the transmission mechanism.
3. **Growth rate amplitudes differ:** Trade oscillates ±20% MoM; revenue shows larger swings early in the period (business ramp-up) but stabilizes after 2023.

These findings direct this notebook toward **growth-rate correlations** as the primary analytical layer, with raw correlations included for completeness and contrast.

### Analytical approach

We test three correlation methods:
- **Pearson:** Measures linear co-movement. Sensitive to outliers.
- **Spearman:** Measures rank-based co-movement. Robust to non-normality and outliers.
- **Cross-correlation at lags −6 to +6:** Tests whether trade flows lead or lag sales by up to 6 months.

Each method is applied to:
- Raw levels (for reference)
- Month-over-month growth rates (primary analysis)

Both Exportaciones and Importaciones are tested — the stronger driver is identified empirically.

### Statistical significance
With only 47 months of data (46 after first-differencing), we have limited statistical power. All correlation coefficients are accompanied by p-values. Results with p > 0.05 are interpreted with caution.

---

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

---
## 2. Configuration

In [ ]:
PROCESSED_DIR  = os.path.join(os.getcwd(), "..", "data", "processed")
OUTPUT_FIGURES = os.path.join(os.getcwd(), "..", "outputs", "figures")
OUTPUT_TABLES  = os.path.join(os.getcwd(), "..", "outputs", "tables")

MAX_LAG = 6  # months in each direction

print(f"Processed dir:  {os.path.normpath(PROCESSED_DIR)}")
print(f"Output figures: {os.path.normpath(OUTPUT_FIGURES)}")

---
## 3. Load Data

In [ ]:
df = pd.read_csv(
    os.path.join(PROCESSED_DIR, "merged_monthly_data_enriched.csv"),
    parse_dates=["Month"],
    decimal=","
)

print(f"Shape: {df.shape}")
print(f"Period: {df['Month'].min().date()} -> {df['Month'].max().date()}")
print(f"Columns: {df.columns.tolist()}")

---
## 4. Pearson and Spearman Correlations

We compute correlations between both trade flows and both sales metrics,  
on raw levels and on growth rates.

In [ ]:
def corr_with_pvalue(x, y, method="pearson"):
    """Compute correlation and p-value between two series, dropping NaN pairs."""
    mask = ~(x.isna() | y.isna())
    x_clean, y_clean = x[mask], y[mask]
    if method == "pearson":
        r, p = stats.pearsonr(x_clean, y_clean)
    else:
        r, p = stats.spearmanr(x_clean, y_clean)
    return round(r, 4), round(p, 4), len(x_clean)

# Define variable pairs to test
pairs = [
    ("exports_musd",  "revenue_mxn",     "Exports vs Revenue (levels)"),
    ("imports_musd",  "revenue_mxn",     "Imports vs Revenue (levels)"),
    ("exports_musd",  "transactions",    "Exports vs Transactions (levels)"),
    ("imports_musd",  "transactions",    "Imports vs Transactions (levels)"),
    ("exports_pct",   "revenue_pct",     "Exports vs Revenue (growth rates)"),
    ("imports_pct",   "revenue_pct",     "Imports vs Revenue (growth rates)"),
    ("exports_pct",   "transactions_pct","Exports vs Transactions (growth rates)"),
    ("imports_pct",   "transactions_pct","Imports vs Transactions (growth rates)"),
]

records = []
for x_col, y_col, label in pairs:
    for method in ["pearson", "spearman"]:
        r, p, n = corr_with_pvalue(df[x_col], df[y_col], method)
        records.append({
            "Pair"      : label,
            "Method"    : method.capitalize(),
            "r"         : r,
            "p_value"   : p,
            "n"         : n,
            "significant": "Yes" if p < 0.05 else "No"
        })

corr_results = pd.DataFrame(records)

print(corr_results.to_string(index=False))

---
## 5. Correlation Summary Heatmap

In [ ]:
# Pivot for heatmap — Pearson on growth rates
growth_cols = ["exports_pct", "imports_pct", "revenue_pct", "transactions_pct"]
labels = ["Exports\nMoM%", "Imports\nMoM%", "Revenue\nMoM%", "Transactions\nMoM%"]

growth_df = df[growth_cols].dropna()
corr_heatmap = growth_df.corr(method="pearson")
corr_heatmap.index   = labels
corr_heatmap.columns = labels

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr_heatmap, dtype=bool))

sns.heatmap(
    corr_heatmap,
    mask=mask,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"shrink": 0.7},
    ax=ax
)

ax.set_title("Pearson Correlation — Growth Rates\n(Trade Flows vs SERSA Sales)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "03_correlation_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 03_correlation_heatmap.png")

---
## 6. Cross-Correlation Analysis

For each trade flow variable vs each sales variable, we compute Pearson cross-correlation  
at lags −6 to +6 months on growth rates.

A positive lag k means the trade variable leads sales by k months:  
corr(trade_t, sales_{t+k})

In [ ]:
def cross_corr(series_a, series_b, max_lag):
    """Cross-correlation at lags -max_lag to +max_lag. Returns dict {lag: (r, p, n)}."""
    results = {}
    for lag in range(-max_lag, max_lag + 1):
        if lag == 0:
            a = series_a.reset_index(drop=True)
            b = series_b.reset_index(drop=True)
        elif lag > 0:
            a = series_a.iloc[:-lag].reset_index(drop=True)
            b = series_b.iloc[lag:].reset_index(drop=True)
        else:
            a = series_a.iloc[-lag:].reset_index(drop=True)
            b = series_b.iloc[:lag].reset_index(drop=True)

        combined = pd.concat([a, b], axis=1).dropna()
        if len(combined) < 5:
            results[lag] = (np.nan, np.nan, 0)
        else:
            r, p = stats.pearsonr(combined.iloc[:, 0], combined.iloc[:, 1])
            results[lag] = (round(r, 4), round(p, 4), len(combined))
    return results

print("cross_corr() defined.")

In [ ]:
# Compute cross-correlations for all combinations
cc_pairs = [
    ("exports_pct", "revenue_pct",      "Exports → Revenue"),
    ("imports_pct", "revenue_pct",      "Imports → Revenue"),
    ("exports_pct", "transactions_pct", "Exports → Transactions"),
    ("imports_pct", "transactions_pct", "Imports → Transactions"),
]

cc_records = []
cc_summary = {}

for x_col, y_col, label in cc_pairs:
    series_a = df[x_col].dropna()
    series_b = df[y_col]
    # align indices
    series_b = series_b.loc[series_a.index]

    cc = cross_corr(series_a, series_b, MAX_LAG)
    cc_summary[label] = cc

    for lag, (r, p, n) in cc.items():
        cc_records.append({
            "Pair"       : label,
            "lag"        : lag,
            "Pearson_r"  : r,
            "p_value"    : p,
            "n"          : n,
            "significant": "Yes" if (p is not None and not np.isnan(p) and p < 0.05) else "No"
        })

cc_df = pd.DataFrame(cc_records)
print(f"Cross-correlation records: {len(cc_df)}")
print()
print(cc_df[cc_df["significant"] == "Yes"].to_string(index=False))

---
## 7. Cross-Correlation Profile Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, (label, cc) in enumerate(cc_summary.items()):
    lags = list(cc.keys())
    rs   = [cc[l][0] if not np.isnan(cc[l][0]) else 0 for l in lags]
    ps   = [cc[l][1] for l in lags]

    colors = ["#D7191C" if r < 0 else "#2C7BB6" for r in rs]
    # darker for significant
    colors = ["#1A5276" if (p is not None and not np.isnan(p) and p < 0.05 and r > 0)
               else "#922B21" if (p is not None and not np.isnan(p) and p < 0.05 and r < 0)
               else c for r, p, c in zip(rs, ps, colors)]

    ax = axes[i]
    ax.bar(lags, rs, color=colors, alpha=0.85)
    ax.axhline(y=0,    color="black", linewidth=0.8)
    ax.axvline(x=0,    color="gray",  linewidth=0.8, linestyle="--", alpha=0.5)
    ax.axhline(y=0.3,  color="orange", linewidth=0.8, linestyle=":", alpha=0.7)
    ax.axhline(y=-0.3, color="orange", linewidth=0.8, linestyle=":", alpha=0.7)
    ax.set_ylim(-1, 1)
    ax.set_xticks(range(-MAX_LAG, MAX_LAG + 1))
    ax.set_xlabel("Lag (months)  [+ = trade leads sales]", fontsize=8)
    ax.set_ylabel("Pearson r", fontsize=9)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    sns.despine(ax=ax)

fig.suptitle("Cross-Correlation Profiles — Trade Flows vs SERSA Sales (Growth Rates)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "03_crosscorr_profiles.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 03_crosscorr_profiles.png")

---
## 8. Dominant Lag per Pair

In [ ]:
dominant_records = []
for label, cc in cc_summary.items():
    valid = {l: cc[l] for l in cc if not np.isnan(cc[l][0])}
    if not valid:
        continue
    dominant_lag = max(valid, key=lambda l: abs(valid[l][0]))
    r, p, n = valid[dominant_lag]
    dominant_records.append({
        "Pair"         : label,
        "dominant_lag" : dominant_lag,
        "max_abs_r"    : abs(r),
        "r_at_lag"     : r,
        "p_value"      : p,
        "significant"  : "Yes" if p < 0.05 else "No",
        "interpretation": (
            "Trade leads sales" if dominant_lag > 0
            else "Sales leads trade" if dominant_lag < 0
            else "Contemporaneous"
        )
    })

dominant_df = pd.DataFrame(dominant_records)
print("Dominant lag per pair:")
print(dominant_df.to_string(index=False))

---
## 9. Export

In [ ]:
corr_results.to_csv(
    os.path.join(OUTPUT_TABLES, "03_correlation_results.csv"),
    index=False, decimal=","
)

cc_df.to_csv(
    os.path.join(OUTPUT_TABLES, "03_crosscorr_results.csv"),
    index=False, decimal=","
)

dominant_df.to_csv(
    os.path.join(OUTPUT_TABLES, "03_dominant_lag_per_pair.csv"),
    index=False, decimal=","
)

print("Export complete.")
print(f"  03_correlation_results.csv   -> {len(corr_results)} rows")
print(f"  03_crosscorr_results.csv     -> {len(cc_df)} rows")
print(f"  03_dominant_lag_per_pair.csv -> {len(dominant_df)} rows")

---
## 10. Summary

| Output | Description |
|--------|-------------|
| `03_correlation_results.csv` | Pearson + Spearman correlations with p-values |
| `03_crosscorr_results.csv` | Full cross-correlation at each lag for all pairs |
| `03_dominant_lag_per_pair.csv` | Dominant lag and max r per pair |
| `03_correlation_heatmap.png` | Heatmap of growth-rate correlations |
| `03_crosscorr_profiles.png` | Cross-correlation bar charts at lags −6 to +6 |

### Key questions answered here
- Is there a statistically significant linear or rank relationship between trade flows and sales?
- Does either trade flow lead or lag sales?
- Is Exports or Imports the stronger driver?

**Next:** `04_product_level_sensitivity.ipynb` — individual SKU correlations against trade flows to identify which products react most strongly to external trade activity.